# 03 — Modélisation Running

**Objectif** : Entraîner un modèle de prédiction de temps marathon depuis les données disponibles, et le comparer à la baseline Riegel.

**Architecture des prédictions :**
- **Mode simple** (pas de temps de référence) : formule Astrand → VO2max → VDOT → temps estimé
- **Mode avancé** (avec temps récent) : modèle ML entraîné sur Boston Marathon

**Dataset** : `data/processed/running_clean.csv`

---

## Plan
1. Chargement des données nettoyées
2. Feature engineering
3. Baseline Riegel (référence)
4. Modèle ML — entraînement et évaluation
5. Comparaison baseline vs ML
6. Analyse des erreurs
7. Export du modèle

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import warnings
from datetime import datetime

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

RANDOM_STATE = 42

## 1. Chargement des données

In [ ]:
df = pd.read_csv('../data/processed/running_clean.csv')
print(f"Dataset : {len(df):,} lignes, {len(df.columns)} colonnes")
print(df.head())

## 2. Feature engineering

In [ ]:
# Features disponibles
# Mode avancé : temps 5K ou 10K + âge + sexe
# Target : temps marathon

# Sélectionner les lignes avec 5K et 10K disponibles
split_cols = [c for c in df.columns if c.endswith('_sec') and c != 'Official Time_sec']
print("Splits disponibles :", split_cols)

# Dataset pour modèle basé sur 5K
df_5k = df[df['5K_sec'].notna() & df['Official Time_sec'].notna()].copy()
print(f"\nLignes avec 5K valide : {len(df_5k):,}")

# Dataset pour modèle basé sur 10K
df_10k = df[df['10K_sec'].notna() & df['Official Time_sec'].notna()].copy()
print(f"Lignes avec 10K valide : {len(df_10k):,}")

# Vitesses dérivées (km/h)
df_5k['speed_5k'] = 5 / (df_5k['5K_sec'] / 3600)
df_10k['speed_10k'] = 10 / (df_10k['10K_sec'] / 3600)

In [ ]:
# Définition des features et target
FEATURES_SIMPLE = ['Age', 'gender']  # Sans temps de référence
FEATURES_5K = ['Age', 'gender', '5K_sec', 'speed_5k']  # Avec temps 5K
FEATURES_10K = ['Age', 'gender', '10K_sec', 'speed_10k']  # Avec temps 10K
TARGET = 'Official Time_sec'

print("Features mode avancé (5K) :", FEATURES_5K)
print("Features mode avancé (10K) :", FEATURES_10K)
print("Target :", TARGET)

## 3. Baseline Riegel

In [ ]:
def riegel(t1_sec, d1_km, d2_km, exponent=1.06):
    return t1_sec * (d2_km / d1_km) ** exponent

def evaluate_predictions(y_true, y_pred, label='Modèle'):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    print(f"{label}:")
    print(f"  MAE  = {mae:.0f}s ({mae/60:.1f} min)")
    print(f"  RMSE = {rmse:.0f}s ({rmse/60:.1f} min)")
    print(f"  MAPE = {mape:.1f}%")
    print(f"  R²   = {r2:.4f}")
    return {'mae': mae, 'rmse': rmse, 'mape': mape, 'r2': r2}

# Baseline 5K → Marathon
df_5k['riegel_marathon'] = df_5k['5K_sec'].apply(lambda t: riegel(t, 5, 42.195))
baseline_5k = evaluate_predictions(df_5k[TARGET], df_5k['riegel_marathon'], 'Baseline Riegel (5K→Marathon)')
print()

# Baseline 10K → Marathon
df_10k['riegel_marathon'] = df_10k['10K_sec'].apply(lambda t: riegel(t, 10, 42.195))
baseline_10k = evaluate_predictions(df_10k[TARGET], df_10k['riegel_marathon'], 'Baseline Riegel (10K→Marathon)')

## 4. Modèle ML — entraînement

In [ ]:
# Split train/test
X = df_5k[FEATURES_5K].dropna()
y = df_5k.loc[X.index, TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print(f"Train : {len(X_train):,} | Test : {len(X_test):,}")

In [ ]:
# Random Forest
rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
metrics_rf = evaluate_predictions(y_test, y_pred_rf, 'Random Forest (5K features)')
print()

In [ ]:
# Gradient Boosting
gb = GradientBoostingRegressor(n_estimators=100, max_depth=4, learning_rate=0.1, random_state=RANDOM_STATE)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)
metrics_gb = evaluate_predictions(y_test, y_pred_gb, 'Gradient Boosting (5K features)')
print()

In [ ]:
# Validation croisée du meilleur modèle
best_model = rf if metrics_rf['mae'] < metrics_gb['mae'] else gb
best_name = 'Random Forest' if metrics_rf['mae'] < metrics_gb['mae'] else 'Gradient Boosting'

cv_scores = cross_val_score(best_model, X, y, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
cv_mae = -cv_scores
print(f"Validation croisée 5-fold ({best_name}) :")
print(f"  MAE par fold : {[f'{v/60:.1f}min' for v in cv_mae]}")
print(f"  MAE moyenne : {cv_mae.mean()/60:.1f} min (±{cv_mae.std()/60:.1f} min)")

## 5. Comparaison Baseline vs ML

In [ ]:
# Comparer sur le même jeu de test
test_data = df_5k.loc[X_test.index].copy()
riegel_on_test = evaluate_predictions(y_test, test_data['riegel_marathon'], 'Baseline Riegel (même test set)')
print()
ml_on_test = evaluate_predictions(y_test, y_pred_rf if best_name == 'Random Forest' else y_pred_gb, f'{best_name} (même test set)')

print(f"\n→ Gain MAE : {(riegel_on_test['mae'] - ml_on_test['mae'])/60:.1f} minutes")
print(f"→ Gain MAPE : {riegel_on_test['mape'] - ml_on_test['mape']:.1f} points")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Prédit vs réel — ML
axes[0].scatter(y_test / 3600, (y_pred_rf if best_name == 'Random Forest' else y_pred_gb) / 3600,
                alpha=0.1, s=5, color='steelblue')
lims = [y_test.min() / 3600, y_test.max() / 3600]
axes[0].plot(lims, lims, 'r--', linewidth=1)
axes[0].set_title(f'{best_name}\nPrédit vs Réel')
axes[0].set_xlabel('Temps réel (h)')
axes[0].set_ylabel('Temps prédit (h)')

# Prédit vs réel — Riegel
axes[1].scatter(y_test / 3600, test_data['riegel_marathon'] / 3600,
                alpha=0.1, s=5, color='coral')
axes[1].plot(lims, lims, 'r--', linewidth=1)
axes[1].set_title('Baseline Riegel\nPrédit vs Réel')
axes[1].set_xlabel('Temps réel (h)')
axes[1].set_ylabel('Temps prédit (h)')

# Feature importance
importances = best_model.feature_importances_
axes[2].barh(FEATURES_5K, importances, color='steelblue')
axes[2].set_title('Feature importance')
axes[2].set_xlabel('Importance')

plt.tight_layout()
plt.savefig('../data/processed/model_running_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Prédiction multi-distances

Le modèle prédit le temps marathon. Les autres distances sont extrapolées via Riegel depuis ce temps prédit.

In [ ]:
def predict_all_distances(model, age, gender, time_5k_sec):
    """
    Prédit les temps sur toutes les distances depuis un profil utilisateur.
    Retourne un dict {distance: temps_sec}.
    """
    speed_5k = 5 / (time_5k_sec / 3600)
    features = pd.DataFrame([[age, gender, time_5k_sec, speed_5k]], columns=FEATURES_5K)
    marathon_sec = model.predict(features)[0]

    distances = {'5km': 5, '10km': 10, 'Semi': 21.0975, 'Marathon': 42.195}
    results = {}
    for name, dist in distances.items():
        if dist == 42.195:
            results[name] = marathon_sec
        else:
            results[name] = riegel(marathon_sec, 42.195, dist)
    return results

def seconds_to_hms(s):
    h = int(s // 3600)
    m = int((s % 3600) // 60)
    sec = int(s % 60)
    if h > 0:
        return f"{h}h{m:02d}m{sec:02d}s"
    return f"{m}m{sec:02d}s"

# Exemple : homme 30 ans, 5K en 22 minutes
example = predict_all_distances(best_model, age=30, gender=0, time_5k_sec=22*60)
print("Exemple : homme 30 ans, 5K en 22:00")
for dist, t in example.items():
    print(f"  {dist:10s} : {seconds_to_hms(t)}")

## 7. Export du modèle

In [ ]:
model_path = '../models/running_model.pkl'
joblib.dump(best_model, model_path)
print(f"Modèle exporté : {model_path}")

metadata = {
    'model_type': best_name,
    'sport': 'running',
    'features': FEATURES_5K,
    'target': TARGET,
    'target_description': 'Temps marathon en secondes',
    'train_samples': len(X_train),
    'test_samples': len(X_test),
    'metrics': {
        'mae_seconds': round(ml_on_test['mae'], 1),
        'mae_minutes': round(ml_on_test['mae'] / 60, 1),
        'mape_pct': round(ml_on_test['mape'], 2),
        'r2': round(ml_on_test['r2'], 4)
    },
    'baseline_riegel': {
        'mae_seconds': round(riegel_on_test['mae'], 1),
        'mae_minutes': round(riegel_on_test['mae'] / 60, 1),
        'mape_pct': round(riegel_on_test['mape'], 2)
    },
    'dataset': 'Boston Marathon 2015-2017',
    'trained_at': datetime.now().isoformat(),
    'limitations': [
        'Dataset Boston Marathon = coureurs qualifiés (temps minimum requis)',
        'Sous-représentation des débutants (> 4h30)',
        'Absence de données physiologiques brutes (poids, FC)',
        'Mode simple (sans temps de référence) = formules VDOT/Astrand uniquement'
    ]
}

with open('../models/running_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print("Métadonnées exportées : models/running_metadata.json")
print()
print(json.dumps(metadata, indent=2, ensure_ascii=False))

## 8. Conclusions

### Résultats
| Modèle | MAE | MAPE | R² |
|---|---|---|---|
| Baseline Riegel | X min | X% | — |
| Random Forest | X min | X% | X |
| Gradient Boosting | X min | X% | X |

*(Valeurs à compléter après exécution)*

### Ce que le modèle fait bien
- Prédit avec précision dans la tranche 3h00-4h30 (majorité du dataset)
- Corrige les biais de la formule Riegel (tendance à surestimer sur marathon)

### Limites
- Coureurs très rapides (< 2h45) et très lents (> 5h) : moins représentés → moins précis
- Mode simple sans temps de référence : prédictions basées uniquement sur formules physiologiques, précision ±15-20%

### Usage dans l'API
- Le modèle reçoit : `[age, gender, 5k_sec, speed_5k]`
- Il retourne : `marathon_sec`
- Les autres distances (5K, 10K, semi) sont calculées via Riegel depuis le marathon prédit